In [1]:
import requests
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
url = "http://ws.audioscrobbler.com/2.0/"
API_KEY = ""

df = pd.read_csv("data/songs.csv")
df.head()

,artist,track,listeners,playcount
0,The Weeknd,Blinding Lights,2314795,37860617
1,The Weeknd,Starboy,2297163,32833915
2,The Weeknd,The Hills,2084094,27009303
3,The Weeknd,Die for You,1954869,33126883
4,The Weeknd,Can't Feel My Face,1826293,18326029


In [2]:
def musicbrainz_genres(artist_name):
    url = f"https://musicbrainz.org/ws/2/artist/?query=artist:{artist_name}&fmt=json"
    data = requests.get(url).json()
    artists = data.get("artists", [])
    if artists:
        return [tag["name"] for tag in artists[0].get("tags", [])]
    return []

In [3]:
def getLabel(artist_name, track_name, index):
    params = {
        "method": "track.getInfo",
        "artist": artist_name,
        "track": track_name,
        "api_key": API_KEY,
        "format": "json"
    }
    track_info = requests.get(url, params=params).json()
    tags =[]
    if "track" in track_info and "toptags" in track_info["track"]:
        tags = [tag["name"] for tag in track_info["track"]["toptags"].get("tag", [])]
    if(len(tags) != 0):
        df.loc[index, 'tags'] = ", ".join(tags)
    else:
        tags = musicbrainz_genres(artist_name)
        time.sleep(1)
        
    print(f"{artist_name}, {track_name}, {tags}")
    time.sleep(.1)
        

In [4]:
import time
import random
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

lock = Lock()
last_call = 0
MIN_INTERVAL = 0.25   # 4 requests/sec total

def safe_getlabel(row):
    global last_call

    for attempt in range(3):  # retry 3 times
        try:
            # rate limit
            with lock:
                now = time.time()
                wait = MIN_INTERVAL - (now - last_call)
                if wait > 0:
                    time.sleep(wait)
                last_call = time.time()

            return getLabel(row.artist, row.track, row.Index)

        except Exception as e:
            if attempt == 2:
                print(f"Failed: {row.artist} - {row.track}")
            time.sleep(2 + random.random())  # backoff

In [ ]:
rows = list(df.itertuples())

with ThreadPoolExecutor(max_workers=10) as executor:
    executor.map(lambda r: getLabel(r.artist, r.track, r.Index), rows)

The Weeknd, Die for You, ['rnb', 'pop', 'Ballad', 'synthpop', 'Canadian']The Weeknd, Can't Feel My Face, ['rnb', 'pop', 'love at first listen', 'r&b', 'Disco']

The Weeknd, Blinding Lights, ['synthwave', 'synthpop', 'pop', '2019', '2010s']
The Weeknd, Stargirl Interlude, ['Lana Del Rey', 'trip hop', 'art pop', 'The Weeknd', 'pop']
The Weeknd, Call Out My Name, ['rnb', 'soul', 'Canadian', 'trap', '2018']
The Weeknd, Save Your Tears, ['synthpop', 'pop', '-1001740215468', 'rnb', 'synthwave']
The Weeknd, I Feel It Coming, ['rnb', 'pop', 'Daft Punk', 'Disco', 'synth funk']
The Weeknd, One of the Girls (with JENNIE, Lily Rose Depp), ['sexy', 'pop', 'Soundtrack', 'synthpop', 'ethereal']
The Weeknd, The Hills, ['rnb', 'electronic', 'trap', 'r&b', 'love at first listen']
The Weeknd, Starboy, ['rnb', 'pop', 'electronic', 'Daft Punk', '2016']
Taylor Swift, Blank Space, ['pop', 'love at first listen', '2014', 'female vocalists', 'best of 2014']
Taylor Swift, Cruel Summer, ['electropop', 'pop', 'sy